In [1]:
import os
import json

import chromadb
import nest_asyncio
nest_asyncio.apply()  # 이벤트 루프 중복 방지(jupyter 환경인 경우)

from dotenv import load_dotenv
load_dotenv()

from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.schema import TextNode
from llama_index.core.prompts import PromptTemplate
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI

# 환경 설정(Google API Key)

In [2]:
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

# 모델 설정(Embedding, LLM)

In [3]:
def setup_model():
    # 모델 설정
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")
    llm = GoogleGenAI(model="gemini-3-flash-preview",
                      api_key=GOOGLE_API_KEY)

    # LlamaIndex 설정
    Settings.llm = llm
    Settings.embed_model = embed_model

    print("임베딩, LLM 모델 설정 완료!")

In [4]:
setup_model()

임베딩, LLM 모델 설정 완료!


# 인덱싱 및 Vector DB 저장

In [5]:
def setup_db(nodes):
    # paper_id 추출
    paper_id = nodes[0].metadata.get("paper_id")
    if paper_id is None:
        raise ValueError("nodes metadata에 paper_id가 없습니다.")
    
    # ChromaDB
    db = chromadb.PersistentClient(path="./chromadb")
    chroma_collection = db.get_or_create_collection("papaer_collection")

    # Vector Store, Storage Context 생성
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    existing = chroma_collection.get(where={"paper_id": paper_id},
                                     include=[])
    existing_ids = set(existing["ids"])

    # 인덱스 생성 및 저장(같은 논문이면 로드, 아니면 생성)
    if len(existing["ids"]) > 0:
        print(f"이미 인덱싱된 논문(paper_id={paper_id}) → 기존 인덱스 로드.")
        index = VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_context)
    else:
        print("새로운 인덱스 생성...", end='')
        index = VectorStoreIndex(nodes, storage_context=storage_context)
        print("저장 완료!")

    return index

In [6]:
# 저장된 노드 불러오기
with open("./nodes.json", 'r', encoding='utf-8') as f:
    node_dicts = json.load(f)

nodes = [TextNode.from_dict(d) for d in node_dicts]

# 인덱싱
index = setup_db(nodes)

이미 인덱싱된 논문(paper_id=5313b60e59bbcb9baa9d087b3b82d5cb64f52a6e9d819c558fd325713b240d4b) → 기존 인덱스 로드.


# 검색 및 답변 테스트

In [7]:
def test_retrieval(index, query):
    print("\n--- [검색 테스트] ---\n")

    # 검색
    retriever = index.as_retriever(similarity_top_k=20)
    retrieved_nodes = retriever.retrieve(query)

    for i, node in enumerate(retrieved_nodes):
        print(f"[검색 결과 {i+1}] (Score: {node.score:.4f})")
        print(f"내용 요약: {node.node.get_content()[:150]}...")

In [8]:
query = "Introduction에서 무엇을 설명하는가"
test_retrieval(index, query)


--- [검색 테스트] ---

[검색 결과 1] (Score: 0.3455)
내용 요약: ## References

- Ahmed, N., Natarajan, T., and Rao, K. R. Discrete cosine transform. IEEE Transactions on Computers , 1974.
- Bai, S., Cai, Y ., Chen,...
[검색 결과 2] (Score: 0.3401)
내용 요약: ## Impact Statement

This paper presents work aimed at advancing the field of machine learning, specifically Vision-Language-Action models for robotic...
[검색 결과 3] (Score: 0.3346)
내용 요약: ## 2.2. The Perception Essentials

In this section, we shift our focus from foundational components to perception, investigating whether and how diffe...
[검색 결과 4] (Score: 0.3296)
내용 요약: ## 5. Conclusion

This work moves toward a more systematic understanding of VLA models. Rather than introducing another standalone architecture, we re...
[검색 결과 5] (Score: 0.3200)
내용 요약: ## C. Revisiting Robot Learning and VLA Models

Robot learning aims to apply machine learning techniques to robotic control, empowering robots to inte...
[검색 결과 6] (Score: 0.3178)
내용 요약: ## 2.3. Actio

In [9]:
def test_response(index, query):
    qa_prompt = PromptTemplate(
        """
        당신은 논문을 설명하는 AI 연구 도우미입니다.
        
        다음 규칙을 반드시 지켜서 답변하세요.
        
        규칙:
        1. 반드시 한국어로 답변하세요.
        2. 제공된 Context 정보만 사용하세요.
        3. 답변에 사용한 정보는 반드시 [번호] 형식으로 출처를 표시하세요.
        4. Context에 없는 내용은 추측하지 말고 "모르겠습니다"라고 답하세요.
        5. 출처 표기시 문단 혹은 섹션 제목만 사용하세요.
        
        답변 형식:
        
        답변:
        <한국어 답변, 출처 포함>
        
        출처:
        [번호] 섹션 또는 문단 제목
        
        Context:
        ---------------------
        {context_str}
        ---------------------
        
        질문:
        {query_str}
        
        답변:
        """
    )
    
    query_engine = index.as_query_engine(similarity_top_k=20,
                                         text_qa_template=qa_prompt)
    response = query_engine.query(query)
    print(response)

In [10]:
test_response(index, query)

답변:
Introduction 섹션에서는 일반 목적 로봇 제어 분야에서 파운데이션 모델의 발전과 Vision-Language-Action (VLA) 모델의 등장을 설명합니다 [1. Introduction]. VLA 모델은 대규모 시각-언어 백본을 활용하여 시각적 관찰과 언어 지침을 로봇 동작으로 직접 매핑하며, 풍부한 시각적 이해와 언어 기반을 상속받아 일반 목적의 언어 조건부 로봇 정책을 위한 확장 가능한 경로를 제공합니다 [1. Introduction].

이 섹션은 VLA 모델의 파이프라인이 VLM과 정책 모듈의 인터페이스 방식, 정책 훈련 방법, 필수 지각 입력 선택, 동작 표현 및 모델링 방식 등 수많은 설계 선택 사항을 포함하고 있음을 지적합니다 [1. Introduction]. 초기 VLA 연구는 아이디어가 풍부하지만 구조가 부족하고, 다양한 프레임워크와 일관성 없는 훈련 및 평가 프로토콜로 인해 어떤 설계 선택이 중요한지 파악하기 어렵다고 언급합니다 [1. Introduction].

이 연구는 통합된 프레임워크 및 평가 프로토콜 하에서 VLA 설계 공간을 체계적으로 재검토하여 이러한 파편화된 설계 공간에 대한 이해를 높이는 것을 목표로 합니다 [1. Introduction]. 연구는 간단한 기준 VLA에서 시작하여 기초 구성 요소, 지각 필수 요소, 동작 모델링 관점의 세 가지 차원을 따라 설계 공간을 체계적으로 탐색합니다 [1. Introduction]. 이러한 연구를 통해 강력한 VLA 모델을 구축하기 위한 12가지 주요 발견을 도출하며, 특히 VLM과 정책 모듈 간의 소프트 연결, VLM 측 고유수용성 입력 조건화, 동작 생성을 시계열 예측 문제로 프레이밍하고 주파수 영역 모델링을 통합하는 것이 효과적이라는 점을 강조합니다 [1. Introduction].

이 연구의 결과는 이러한 설계 원칙에서 직접 파생된 간단하면서도 효과적인 VLA 모델인 VLANeXt이며, 이는 LIBERO 및 LIBERO-plus 벤치마크에서 최첨단 성능을 달성하